In [1]:
import sys
import os
import warnings
import matplotlib.pyplot as plt
import pandas as pd
import get_silk_genes as silk
from id_converter import convert_hogs_to_locs

pd.set_option('display.max_columns', None, 'display.max_rows', 50)

# Ensure the src directory is in the Python path
scripts = os.path.abspath('')
src_path = os.path.abspath(os.path.join(scripts, '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from hyphy_results_parser import (
    RelaxResult,
    BustedPhResult
)

from odds_ratio_test import BootstrapTestResults

# Define paths to data and results directories
data = os.path.join(scripts, "..", "data/")
figures = os.path.join(scripts, "..", "figures/")
results = os.path.join(scripts, "..", "results/")
assets = os.path.join(scripts, "..", "assets")


/Users/calvin/anaconda3/envs/orb-selection/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def move_column(df, col_name, new_pos):
    cols = list(df.columns)
    cols.insert(new_pos, cols.pop(cols.index(col_name)))
    return df[cols]

## Hyphy Results

In [3]:
hyphy_results = results + "/hyphy_results_cache/"

# Load the saved RELAX results
relax_result = RelaxResult.load_from_pickle(str(hyphy_results + "relax_results.pkl"))
relax_result_fltrd = relax_result.filter_omega(10000)
relax_hits_df = relax_result_fltrd.get_significant_results()
relax_hits_df = move_column(relax_hits_df, 'result', 0)  # Move 'result' column to the front
relax_hits_df['foreground'] = 'non-orbweavers'
relax_hits_df = move_column(relax_hits_df, 'foreground', 1)  # Move 'foreground' column to the front
relax_hits_df

,result,foreground,p_value,k,LRT,MG94xREV_ω_reference,MG94xREV_ω_test,ω1_test,ω1_test_P,ω2_test,ω2_test_P,ω3_test,ω3_test_P,ω1_ref,ω1_ref_P,ω2_ref,ω2_ref_P,ω3_ref,ω3_ref_P,ω_mean_test,ω_mean_ref
HOG,,,,,,,,,,,,,,,,,,,,,
N5.HOG0067228,relaxed,non-orbweavers,5.069278e-13,0.031330,52.178060,2.038968e-05,1.110457e+00,0.000000,0.843804,0.389555,0.063292,1.000000,0.092904,0.000000,0.843804,8.541217e-14,0.063292,1.000000,0.092904,0.117560,0.092904
N5.HOG0059719,intensified,non-orbweavers,7.772996e-06,1.108589,19.992907,7.502321e-02,6.447394e-02,0.037770,0.404703,0.046494,0.595297,1.964199,0.000000,0.052061,0.404703,6.279584e-02,0.595297,1.838516,0.000000,0.042963,0.058452
N5.HOG0063256,intensified,non-orbweavers,7.733957e-04,1.136232,11.304017,6.619584e-02,5.929911e-02,0.000000,0.810437,0.086857,0.189563,1.002673,0.000000,0.000000,0.810437,1.164229e-01,0.189563,1.002352,0.000000,0.016465,0.022069
N5.HOG0053197,intensified,non-orbweavers,1.414873e-05,1.212468,18.848973,1.444946e-07,1.613655e-07,0.049339,0.277338,0.065543,0.722662,1.003152,0.000000,0.083597,0.277338,1.056603e-01,0.722662,1.002599,0.000000,0.061049,0.099541
N5.HOG0038544,intensified,non-orbweavers,7.216346e-05,1.097131,15.753260,2.865946e+00,4.260989e-06,0.053668,0.706817,0.054859,0.293183,1.002547,0.000000,0.069531,0.706817,7.093601e-02,0.293183,1.002321,0.000000,0.054018,0.069943
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
N5.HOG0067687,intensified,non-orbweavers,9.106801e-04,1.152606,11.000893,1.822100e-01,1.666336e-01,0.099776,0.262095,0.107294,0.737905,2.158247,0.000000,0.135381,0.262095,1.441878e-01,0.737905,1.949243,0.000000,0.105324,0.141880
N5.HOG0064241,intensified,non-orbweavers,2.168763e-02,1.117584,5.270673,5.236016e-05,4.930010e-01,0.000000,0.302399,0.152434,0.697601,1.862849,0.000000,0.000000,0.302399,1.857939e-01,0.697601,1.744823,0.000000,0.106338,0.129610
N5.HOG0066936,intensified,non-orbweavers,6.469430e-05,1.122134,15.960024,1.096739e-07,8.254460e-01,0.012756,0.285540,0.088955,0.714460,1.001747,0.000000,0.020506,0.285540,1.157564e-01,0.714460,1.001557,0.000000,0.067197,0.088559


In [4]:
# Load the saved BUSTED-PH results
busted_ph_result = BustedPhResult.load_from_pickle(str(hyphy_results + "busted_ph_results.pkl"))
busted_ph_result_fltrd = busted_ph_result.filter_omega(10000)
busted_ph_hits_df = busted_ph_result_fltrd.get_significant_results()
busted_ph_hits_df['foreground'] = 'orbweavers'
busted_ph_hits_df = move_column(busted_ph_hits_df, 'foreground', 0)  # Move 'foreground' column to the front
busted_ph_hits_df.drop(columns=['result'], inplace=True)  # Drop result column
busted_ph_hits_df

/var/folders/h8/wkfyd5dj3nnd_smftqfplclh0000gn/T/ipykernel_73349/2871403343.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  busted_ph_hits_df['foreground'] = 'orbweavers'


,foreground,test_pval,test_LRT,background_pval,background_LRT,shared_pval,shared_LRT,MG94xREV_ω_test,MG94xREV_ω_ref,ω1_test,ω1_test_P,ω2_test,ω2_test_P,ω3_test,ω3_test_P,ω1_ref,ω1_ref_P,ω2_ref,ω2_ref_P,ω3_ref,ω3_ref_P,ω_mean_test,ω_mean_ref
HOG,,,,,,,,,,,,,,,,,,,,,,,
N5.HOG0058262,orbweavers,1.545704e-03,11.558257,0.194303,1.890383,2.477354e-02,12.855262,0.194701,1.595677e-01,0.000000,0.561941,0.187650,0.437508,41.105141,0.000551,0.000000,0.425298,0.108554,0.561794,1.974842,0.012908,0.104760,0.086476
N5.HOG0059024,orbweavers,1.246802e-03,11.988053,0.500000,0.000000,1.286804e-02,14.473037,0.473107,1.181737e-07,0.000000,0.270546,0.185910,0.702125,3.052891,0.027329,0.064841,0.000000,0.065182,0.898490,1.000000,0.101510,0.213964,0.160075
N5.HOG0062008,orbweavers,4.821284e-08,32.308987,0.408194,0.405731,2.667662e-07,38.749055,0.308877,2.457620e-01,0.000000,0.097881,0.159755,0.897586,48.943871,0.004533,0.062739,0.753940,0.082826,0.189299,1.319449,0.056761,0.365235,0.137874
N5.HOG0032852,orbweavers,1.303768e-11,48.740062,0.477842,0.090656,9.747535e-04,20.573963,0.154262,1.945747e-01,0.009604,0.626402,0.217712,0.372822,3967.329535,0.000775,0.001545,0.069786,0.080352,0.886788,1.120800,0.043426,3.162785,0.120035
N5.HOG0060473,orbweavers,7.599450e-08,31.398915,0.500000,0.000000,4.055495e-04,22.582696,0.239854,2.098764e-01,0.044892,0.586331,0.490225,0.412124,510139.703369,0.001545,0.001943,0.318434,0.109385,0.427393,0.623890,0.254173,788.217417,0.205945
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
N5.HOG0059005,orbweavers,0.000000e+00,205.496307,0.500000,0.000000,1.873424e-07,39.511472,0.316642,2.354017e-01,0.001939,0.451225,0.473936,0.502813,19.647255,0.045962,0.000000,0.602167,0.000579,0.000000,0.555289,0.397833,1.142205,0.220912
N5.HOG0056667,orbweavers,3.393248e-08,33.011492,0.500000,0.000000,5.346662e-06,32.231194,0.157910,1.543083e-01,0.131782,0.991548,0.133366,0.006880,160.277082,0.001573,0.007861,0.328270,0.083886,0.489142,0.452285,0.182587,0.383626,0.126194
N5.HOG0068073,orbweavers,4.157678e-04,14.184473,0.500000,0.000000,1.621171e-03,19.396446,0.231503,2.064418e-01,0.026364,0.018939,0.210999,0.978294,455.127141,0.002767,0.099528,0.855957,0.218815,0.036574,1.000000,0.107469,1.466394,0.200663


In [5]:
# Load the saved BUSTED-PH-rev results
busted_ph_rev_result = BustedPhResult.load_from_pickle(str(hyphy_results + "busted_ph_rev_results.pkl"))
busted_ph_rev_result_fltrd = busted_ph_rev_result.filter_omega(10000)
busted_ph_rev_hits_df = busted_ph_rev_result_fltrd.get_significant_results()
busted_ph_rev_hits_df['foreground'] = 'non-orbweavers'
busted_ph_rev_hits_df = move_column(busted_ph_rev_hits_df, 'foreground', 0)  # Move 'foreground' column to the front
busted_ph_rev_hits_df.drop(columns=['result'], inplace=True)  # Drop result column
busted_ph_rev_hits_df

/var/folders/h8/wkfyd5dj3nnd_smftqfplclh0000gn/T/ipykernel_73349/1011142091.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  busted_ph_rev_hits_df['foreground'] = 'non-orbweavers'


,foreground,test_pval,test_LRT,background_pval,background_LRT,shared_pval,shared_LRT,MG94xREV_ω_test,MG94xREV_ω_ref,ω1_test,ω1_test_P,ω2_test,ω2_test_P,ω3_test,ω3_test_P,ω1_ref,ω1_ref_P,ω2_ref,ω2_ref_P,ω3_ref,ω3_ref_P,ω_mean_test,ω_mean_ref
HOG,,,,,,,,,,,,,,,,,,,,,,,
N5.HOG0019712,non-orbweavers,0.000000e+00,809.032492,0.500000,0.000000,0.000000e+00,449.467708,0.000040,0.000017,0.028944,0.645432,0.312607,0.330121,182.484806,0.024447,0.000000,0.071261,0.014606,0.826985,1.000000,0.101753,4.583093,0.113832
N5.HOG0056026,non-orbweavers,8.155119e-10,40.468116,0.500000,0.000000,5.973391e-04,21.698708,0.135780,0.128210,0.051367,0.857685,0.500453,0.139372,5258.822922,0.002943,0.008322,0.719924,0.185418,0.194951,0.859716,0.085125,15.589621,0.115322
N5.HOG0062863,non-orbweavers,3.469447e-14,60.596936,0.500000,0.000000,3.956550e-04,22.638929,0.211636,0.162953,0.081360,0.987843,0.738062,0.007109,4553.459205,0.005048,0.000000,0.068601,0.038564,0.818016,0.530455,0.113383,23.070581,0.091690
N5.HOG0041381,non-orbweavers,0.000000e+00,102.987142,0.109140,3.043957,6.191836e-06,31.909387,0.166545,0.161066,0.000000,0.006265,0.133910,0.987070,25007.129413,0.006665,0.000000,0.381828,0.155732,0.584634,1.691232,0.033538,166.802487,0.147767
N5.HOG0036458,non-orbweavers,2.299926e-02,6.158292,0.133744,2.637359,3.273738e-04,23.069636,0.005095,0.007983,0.001590,0.904070,0.003990,0.095343,4872.117587,0.000587,0.003572,0.994320,0.003753,0.003940,2.572762,0.001740,2.860019,0.008043
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
N5.HOG0067033,non-orbweavers,0.000000e+00,104.322215,0.500000,-20.557023,0.000000e+00,111.260119,0.000032,0.000035,0.000000,0.461302,0.166979,0.534538,178169.075985,0.004160,0.000000,0.485872,0.158264,0.494977,1.948169,0.019151,741.286697,0.115647
N5.HOG0065829,non-orbweavers,0.000000e+00,197.638764,0.500000,-844.964242,2.065015e-11,58.885575,0.000105,0.000077,0.003391,0.582894,0.483803,0.369519,13.393360,0.047587,0.000000,0.282822,0.045675,0.507407,1.848422,0.209771,0.818094,0.410921
N5.HOG0067927,non-orbweavers,0.000000e+00,146.664449,0.500000,0.000000,4.634104e-11,57.183980,0.192987,0.207552,0.035514,0.893746,0.701318,0.089573,6248.594546,0.016681,0.000000,0.030754,0.031915,0.809310,0.728830,0.159936,104.326700,0.142395


In [6]:
all_busted_hits_df = pd.concat([busted_ph_hits_df, busted_ph_rev_hits_df])
all_busted_hits_df

,foreground,test_pval,test_LRT,background_pval,background_LRT,shared_pval,shared_LRT,MG94xREV_ω_test,MG94xREV_ω_ref,ω1_test,ω1_test_P,ω2_test,ω2_test_P,ω3_test,ω3_test_P,ω1_ref,ω1_ref_P,ω2_ref,ω2_ref_P,ω3_ref,ω3_ref_P,ω_mean_test,ω_mean_ref
HOG,,,,,,,,,,,,,,,,,,,,,,,
N5.HOG0058262,orbweavers,1.545704e-03,11.558257,0.194303,1.890383,2.477354e-02,12.855262,0.194701,1.595677e-01,0.000000,0.561941,0.187650,0.437508,41.105141,0.000551,0.000000,0.425298,0.108554,0.561794,1.974842,0.012908,0.104760,0.086476
N5.HOG0059024,orbweavers,1.246802e-03,11.988053,0.500000,0.000000,1.286804e-02,14.473037,0.473107,1.181737e-07,0.000000,0.270546,0.185910,0.702125,3.052891,0.027329,0.064841,0.000000,0.065182,0.898490,1.000000,0.101510,0.213964,0.160075
N5.HOG0062008,orbweavers,4.821284e-08,32.308987,0.408194,0.405731,2.667662e-07,38.749055,0.308877,2.457620e-01,0.000000,0.097881,0.159755,0.897586,48.943871,0.004533,0.062739,0.753940,0.082826,0.189299,1.319449,0.056761,0.365235,0.137874
N5.HOG0032852,orbweavers,1.303768e-11,48.740062,0.477842,0.090656,9.747535e-04,20.573963,0.154262,1.945747e-01,0.009604,0.626402,0.217712,0.372822,3967.329535,0.000775,0.001545,0.069786,0.080352,0.886788,1.120800,0.043426,3.162785,0.120035
N5.HOG0060473,orbweavers,7.599450e-08,31.398915,0.500000,0.000000,4.055495e-04,22.582696,0.239854,2.098764e-01,0.044892,0.586331,0.490225,0.412124,510139.703369,0.001545,0.001943,0.318434,0.109385,0.427393,0.623890,0.254173,788.217417,0.205945
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
N5.HOG0067033,non-orbweavers,0.000000e+00,104.322215,0.500000,-20.557023,0.000000e+00,111.260119,0.000032,3.549836e-05,0.000000,0.461302,0.166979,0.534538,178169.075985,0.004160,0.000000,0.485872,0.158264,0.494977,1.948169,0.019151,741.286697,0.115647
N5.HOG0065829,non-orbweavers,0.000000e+00,197.638764,0.500000,-844.964242,2.065015e-11,58.885575,0.000105,7.708236e-05,0.003391,0.582894,0.483803,0.369519,13.393360,0.047587,0.000000,0.282822,0.045675,0.507407,1.848422,0.209771,0.818094,0.410921
N5.HOG0067927,non-orbweavers,0.000000e+00,146.664449,0.500000,0.000000,4.634104e-11,57.183980,0.192987,2.075518e-01,0.035514,0.893746,0.701318,0.089573,6248.594546,0.016681,0.000000,0.030754,0.031915,0.809310,0.728830,0.159936,104.326700,0.142395


In [7]:
busted_hits_locs = convert_hogs_to_locs(all_busted_hits_df, f'{assets}/N5.tsv')
busted_hits_locs_fltrd = busted_hits_locs.drop(columns=['udiv_genes', 'dmel_orthologs', 'txpt']).drop_duplicates().reset_index(drop=True)
busted_hits_locs_fltrd = move_column(busted_hits_locs_fltrd, 'LOC', 1) 
busted_hits_locs_fltrd.rename(columns={'LOC': 'U_diversus_LOC'}, inplace=True)
busted_hits_locs_fltrd

Processing HOGs: 100%|██████████| 292/292 [00:00<00:00, 1281.09it/s]



,HOG,U_diversus_LOC,foreground,test_pval,test_LRT,background_pval,background_LRT,shared_pval,shared_LRT,MG94xREV_ω_test,MG94xREV_ω_ref,ω1_test,ω1_test_P,ω2_test,ω2_test_P,ω3_test,ω3_test_P,ω1_ref,ω1_ref_P,ω2_ref,ω2_ref_P,ω3_ref,ω3_ref_P,ω_mean_test,ω_mean_ref,GO_terms,Description
0,N5.HOG0058262,129215997,orbweavers,1.545704e-03,11.558257,0.194303,1.890383,2.477354e-02,12.855262,0.194701,1.595677e-01,0.000000,0.561941,0.187650,0.437508,41.105141,0.000551,0.000000,0.425298,0.108554,0.561794,1.974842,0.012908,0.104760,0.086476,GO:0006470 [Name: protein dephosphorylation]; ...,TGF-beta-activated kinase 1 and MAP3K7-binding...
1,N5.HOG0059024,129220007,orbweavers,1.246802e-03,11.988053,0.500000,0.000000,1.286804e-02,14.473037,0.473107,1.181737e-07,0.000000,0.270546,0.185910,0.702125,3.052891,0.027329,0.064841,0.000000,0.065182,0.898490,1.000000,0.101510,0.213964,0.160075,GO:0045944 [Name: positive regulation of trans...,RING finger protein 10-like
2,N5.HOG0062008,129222444,orbweavers,4.821284e-08,32.308987,0.408194,0.405731,2.667662e-07,38.749055,0.308877,2.457620e-01,0.000000,0.097881,0.159755,0.897586,48.943871,0.004533,0.062739,0.753940,0.082826,0.189299,1.319449,0.056761,0.365235,0.137874,GO:0001405 [Name: presequence translocase-asso...,"grpE protein homolog, mitochondrial Roe1"
3,N5.HOG0032852,129218644,orbweavers,1.303768e-11,48.740062,0.477842,0.090656,9.747535e-04,20.573963,0.154262,1.945747e-01,0.009604,0.626402,0.217712,0.372822,3967.329535,0.000775,0.001545,0.069786,0.080352,0.886788,1.120800,0.043426,3.162785,0.120035,GO:0000407 [Name: pre-autophagosomal structure...,autophagy-related protein 13-like
4,N5.HOG0060473,129231803,orbweavers,7.599450e-08,31.398915,0.500000,0.000000,4.055495e-04,22.582696,0.239854,2.098764e-01,0.044892,0.586331,0.490225,0.412124,510139.703369,0.001545,0.001943,0.318434,0.109385,0.427393,0.623890,0.254173,788.217417,0.205945,GO:0005515 [Name: protein binding],leucine-rich repeat-containing protein 59-like
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
318,N5.HOG0067033,129225838,non-orbweavers,0.000000e+00,104.322215,0.500000,-20.557023,0.000000e+00,111.260119,0.000032,3.549836e-05,0.000000,0.461302,0.166979,0.534538,178169.075985,0.004160,0.000000,0.485872,0.158264,0.494977,1.948169,0.019151,741.286697,0.115647,GO:0042273 [Name: ribosomal large subunit biog...,nucleolar complex protein 2 homolog
319,N5.HOG0065829,NaN,non-orbweavers,0.000000e+00,197.638764,0.500000,-844.964242,2.065015e-11,58.885575,0.000105,7.708236e-05,0.003391,0.582894,0.483803,0.369519,13.393360,0.047587,0.000000,0.282822,0.045675,0.507407,1.848422,0.209771,0.818094,0.410921,NaN,NaN
320,N5.HOG0067927,129223042,non-orbweavers,0.000000e+00,146.664449,0.500000,0.000000,4.634104e-11,57.183980,0.192987,2.075518e-01,0.035514,0.893746,0.701318,0.089573,6248.594546,0.016681,0.000000,0.030754,0.031915,0.809310,0.728830,0.159936,104.326700,0.142395,GO:0000981 [Name: RNA polymerase II transcript...,uncharacterized LOC129223042
321,N5.HOG0063651,129220117,non-orbweavers,2.807126e-05,19.575234,0.500000,0.000000,1.631340e-05,29.777420,0.000025,3.369798e-05,0.022216,0.824099,0.370172,0.170712,21.378347,0.005190,0.000000,0.540813,0.212053,0.376480,0.853517,0.082707,0.192446,0.150425,-,uncharacterized LOC129220117


In [8]:
relax_hits_locs = convert_hogs_to_locs(relax_hits_df, f'{assets}/N5.tsv')
relax_hits_locs_fltrd = relax_hits_locs.drop(columns=['udiv_genes', 'dmel_orthologs', 'txpt']).drop_duplicates().reset_index(drop=True)
relax_hits_locs_fltrd = move_column(relax_hits_locs_fltrd, 'LOC', 1) 
relax_hits_locs_fltrd.rename(columns={'LOC': 'U_diversus_LOC'}, inplace=True)
relax_hits_locs_fltrd

Processing HOGs: 100%|██████████| 1429/1429 [00:01<00:00, 1237.77it/s]



,HOG,U_diversus_LOC,result,foreground,p_value,k,LRT,MG94xREV_ω_reference,MG94xREV_ω_test,ω1_test,ω1_test_P,ω2_test,ω2_test_P,ω3_test,ω3_test_P,ω1_ref,ω1_ref_P,ω2_ref,ω2_ref_P,ω3_ref,ω3_ref_P,ω_mean_test,ω_mean_ref,GO_terms,Description
0,N5.HOG0067228,129223414,relaxed,non-orbweavers,5.069278e-13,0.031330,52.178060,2.038968e-05,1.110457e+00,0.000000,0.843804,0.389555,0.063292,1.000000,0.092904,0.000000,0.843804,8.541217e-14,0.063292,1.000000,0.092904,0.117560,0.092904,GO:0005737 [Name: cytoplasm]; GO:0016925 [Name...,activator of SUMO 1
1,N5.HOG0059719,129232318,intensified,non-orbweavers,7.772996e-06,1.108589,19.992907,7.502321e-02,6.447394e-02,0.037770,0.404703,0.046494,0.595297,1.964199,0.000000,0.052061,0.404703,6.279584e-02,0.595297,1.838516,0.000000,0.042963,0.058452,GO:0005634 [Name: nucleus]; GO:0019264 [Name: ...,serine hydroxymethyltransferase-like
2,N5.HOG0059719,129227349,intensified,non-orbweavers,7.772996e-06,1.108589,19.992907,7.502321e-02,6.447394e-02,0.037770,0.404703,0.046494,0.595297,1.964199,0.000000,0.052061,0.404703,6.279584e-02,0.595297,1.838516,0.000000,0.042963,0.058452,GO:0005737 [Name: cytoplasm]; GO:0042803 [Name...,serine hydroxymethyltransferase-like
3,N5.HOG0063256,129224590,intensified,non-orbweavers,7.733957e-04,1.136232,11.304017,6.619584e-02,5.929911e-02,0.000000,0.810437,0.086857,0.189563,1.002673,0.000000,0.000000,0.810437,1.164229e-01,0.189563,1.002352,0.000000,0.016465,0.022069,GO:0003924 [Name: GTPase activity]; GO:0005737...,GPN-loop GTPase 2-like
4,N5.HOG0053197,129220170,intensified,non-orbweavers,1.414873e-05,1.212468,18.848973,1.444946e-07,1.613655e-07,0.049339,0.277338,0.065543,0.722662,1.003152,0.000000,0.083597,0.277338,1.056603e-01,0.722662,1.002599,0.000000,0.061049,0.099541,GO:0005634 [Name: nucleus]; GO:0032502 [Name: ...,molecular chaperone MKKS-like
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1571,N5.HOG0067687,NaN,intensified,non-orbweavers,9.106801e-04,1.152606,11.000893,1.822100e-01,1.666336e-01,0.099776,0.262095,0.107294,0.737905,2.158247,0.000000,0.135381,0.262095,1.441878e-01,0.737905,1.949243,0.000000,0.105324,0.141880,NaN,NaN
1572,N5.HOG0064241,129227186,intensified,non-orbweavers,2.168763e-02,1.117584,5.270673,5.236016e-05,4.930010e-01,0.000000,0.302399,0.152434,0.697601,1.862849,0.000000,0.000000,0.302399,1.857939e-01,0.697601,1.744823,0.000000,0.106338,0.129610,"GO:0006352 [Name: DNA-templated transcription,...",TATA box-binding protein-like 1
1573,N5.HOG0066936,129234669,intensified,non-orbweavers,6.469430e-05,1.122134,15.960024,1.096739e-07,8.254460e-01,0.012756,0.285540,0.088955,0.714460,1.001747,0.000000,0.020506,0.285540,1.157564e-01,0.714460,1.001557,0.000000,0.067197,0.088559,GO:0009058 [Name: biosynthetic process]; GO:00...,eukaryotic translation initiation factor 2B su...
1574,N5.HOG0060161,129226064,intensified,non-orbweavers,1.160829e-03,1.145080,10.551673,5.080422e-02,6.864197e-02,0.000000,0.876649,1.000000,0.118220,2318.115008,0.005132,0.000000,0.876649,1.000000e+00,0.118220,868.514944,0.005132,12.013916,4.575112,GO:0005515 [Name: protein binding]; GO:0007098...,partitioning defective protein 6


In [9]:
relax_hits_locs_fltrd.to_excel(f'{results}/Supplementary_Table_2_RELAX_hits.xlsx', index=False)
busted_hits_locs_fltrd.to_excel(f'{results}/Supplementary_Table_3_BUSTED_hits.xlsx', index=False)

## Bootstrap results

In [10]:
ort_results = results + "/odds_ratio_test/occ_50/"

loss_bsr = BootstrapTestResults.load_from_pickle(str(ort_results + "loss_LT/loss_occ50-95_LT.pkl"))
dup_bsr = BootstrapTestResults.load_from_pickle(str(ort_results + "duplication_RT/duplication_occ50-98_RT.pkl"))

In [11]:
loss_results_df = loss_bsr.results_fltrd_df
loss_results_df.rename(columns={'Odds loss, foreground': 'Odds, orb-weavers', 'Odds loss, background': 'Odds, non-orbweavers'}, inplace=True)
loss_results_df['Test'] = 'Loss'
loss_results_df = move_column(loss_results_df, 'Test', 0)
loss_orb_df = loss_results_df[loss_results_df["Log odds ratio"] > 0]
loss_orb_df['Result'] = 'More likely in orbweavers'

loss_nonorb_df = loss_results_df[loss_results_df["Log odds ratio"] < 0]
loss_nonorb_df['Result'] = 'More likely in non-orbweavers'
loss_df = pd.concat([loss_orb_df, loss_nonorb_df])
loss_df = move_column(loss_df, 'Result', 1)
loss_df

/var/folders/h8/wkfyd5dj3nnd_smftqfplclh0000gn/T/ipykernel_73349/1746798664.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  loss_orb_df['Result'] = 'More likely in orbweavers'
/var/folders/h8/wkfyd5dj3nnd_smftqfplclh0000gn/T/ipykernel_73349/1746798664.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  loss_nonorb_df['Result'] = 'More likely in non-orbweavers'


,Test,Result,Occupancy,"Odds, orb-weavers","Odds, non-orbweavers",Log odds ratio
HOG,,,,,,
N5.HOG0002072,Loss,More likely in orbweavers,52.0,5.923077,0.182796,3.478242
N5.HOG0002341,Loss,More likely in orbweavers,62.0,1.432432,0.235955,1.803488
N5.HOG0004537,Loss,More likely in orbweavers,62.0,1.727273,0.182796,2.245930
N5.HOG0005532,Loss,More likely in orbweavers,85.0,0.384615,0.028037,2.618705
N5.HOG0006029,Loss,More likely in orbweavers,52.0,2.103448,0.428571,1.590876
...,...,...,...,...,...,...
N5.HOG0070810,Loss,More likely in non-orbweavers,53.0,0.232877,2.142857,-2.219386
N5.HOG0070869,Loss,More likely in non-orbweavers,51.0,0.384615,1.820513,-1.554630
N5.HOG0071249,Loss,More likely in non-orbweavers,61.0,0.084337,1.682927,-2.993465


In [12]:
dup_results_df = dup_bsr.results_fltrd_df
dup_results_df.rename(columns={'Odds duplication, foreground': 'Odds, orb-weavers', 'Odds duplication, background': 'Odds, non-orbweavers'}, inplace=True)
dup_results_df['Test'] = 'Duplication'
dup_results_df = move_column(dup_results_df, 'Test', 0)
dup_orb_df = dup_results_df[dup_results_df["Log odds ratio"] > 0]
dup_orb_df['Result'] = 'More likely in orbweavers'

dup_nonorb_df = dup_results_df[dup_results_df["Log odds ratio"] < 0]
dup_nonorb_df['Result'] = 'More likely in non-orbweavers'
dup_df = pd.concat([dup_orb_df, dup_nonorb_df])
dup_df = move_column(dup_df, 'Result', 1)
dup_df

/var/folders/h8/wkfyd5dj3nnd_smftqfplclh0000gn/T/ipykernel_73349/1201252767.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dup_orb_df['Result'] = 'More likely in orbweavers'
/var/folders/h8/wkfyd5dj3nnd_smftqfplclh0000gn/T/ipykernel_73349/1201252767.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dup_nonorb_df['Result'] = 'More likely in non-orbweavers'


,Test,Result,Occupancy,"Odds, orb-weavers","Odds, non-orbweavers",Log odds ratio
HOG,,,,,,
N5.HOG0002083,Duplication,More likely in orbweavers,68.0,0.764706,0.047619,2.776258
N5.HOG0003890,Duplication,More likely in orbweavers,78.0,0.139241,0.028037,1.602664
N5.HOG0004508,Duplication,More likely in orbweavers,51.0,0.267606,0.047619,1.726282
N5.HOG0005150,Duplication,More likely in orbweavers,55.0,0.836735,0.157895,1.667578
N5.HOG0005597,Duplication,More likely in orbweavers,93.0,29.000000,1.972973,2.687754
...,...,...,...,...,...,...
N5.HOG0070689,Duplication,More likely in non-orbweavers,66.0,0.011236,0.089109,-2.070740
N5.HOG0070693,Duplication,More likely in non-orbweavers,60.0,0.034483,0.182796,-1.667910
N5.HOG0071650,Duplication,More likely in non-orbweavers,57.0,0.011236,0.067961,-1.799818


In [13]:
dup_locs = convert_hogs_to_locs(dup_df, f'{assets}/N5.tsv')
dup_locs_filtered = dup_locs.drop(columns=['udiv_genes', 'dmel_orthologs', 'txpt']).drop_duplicates().reset_index(drop=True)
dup_locs_filtered = move_column(dup_locs_filtered, 'LOC', 1)
dup_locs_filtered.rename(columns={'LOC': 'U_diversus_LOC'}, inplace=True)
dup_locs_filtered


Processing HOGs: 100%|██████████| 622/622 [00:00<00:00, 1328.26it/s]


,HOG,U_diversus_LOC,Test,Result,Occupancy,"Odds, orb-weavers","Odds, non-orbweavers",Log odds ratio,GO_terms,Description
0,N5.HOG0002083,129231815,Duplication,More likely in orbweavers,68.0,0.764706,0.047619,2.776258,GO:0008010 [Name: structural constituent of ch...,uncharacterized LOC129231815
1,N5.HOG0003890,129219841,Duplication,More likely in orbweavers,78.0,0.139241,0.028037,1.602664,GO:0005506 [Name: iron ion binding]; GO:002003...,cytochrome P450 3A8-like
2,N5.HOG0004508,129220187,Duplication,More likely in orbweavers,51.0,0.267606,0.047619,1.726282,GO:0000281 [Name: mitotic cytokinesis]; GO:004...,Mitotic spindle and nuclear protein
3,N5.HOG0004508,129220188,Duplication,More likely in orbweavers,51.0,0.267606,0.047619,1.726282,GO:0004867 [Name: serine-type endopeptidase in...,actinia tenebrosa protease inhibitors-like
4,N5.HOG0005150,129227954,Duplication,More likely in orbweavers,55.0,0.836735,0.157895,1.667578,GO:0005615 [Name: extracellular space]; GO:003...,carboxypeptidase N subunit 2-like
...,...,...,...,...,...,...,...,...,...,...
691,N5.HOG0070689,129218512,Duplication,More likely in non-orbweavers,66.0,0.011236,0.089109,-2.070740,-,uncharacterized LOC129218512
692,N5.HOG0070693,129218704,Duplication,More likely in non-orbweavers,60.0,0.034483,0.182796,-1.667910,-,uncharacterized LOC129218704
693,N5.HOG0071650,129217430,Duplication,More likely in non-orbweavers,57.0,0.011236,0.067961,-1.799818,-,uncharacterized LOC129217430
694,N5.HOG0071855,129230365,Duplication,More likely in non-orbweavers,50.0,0.011236,0.067961,-1.799818,GO:0005515 [Name: protein binding],protein artichoke-like


In [14]:
loss_locs = convert_hogs_to_locs(loss_df, f'{assets}/N5.tsv')
loss_locs_filtered = loss_locs.drop(columns=['udiv_genes', 'dmel_orthologs', 'txpt']).drop_duplicates().reset_index(drop=True)
loss_locs_filtered = move_column(loss_locs_filtered, 'LOC', 1)
loss_locs_filtered.rename(columns={'LOC': 'U_diversus_LOC'}, inplace=True)
loss_locs_filtered


Processing HOGs: 100%|██████████| 1412/1412 [00:01<00:00, 1298.56it/s]


,HOG,U_diversus_LOC,Test,Result,Occupancy,"Odds, orb-weavers","Odds, non-orbweavers",Log odds ratio,GO_terms,Description
0,N5.HOG0002072,129217362,Loss,More likely in orbweavers,52.0,5.923077,0.182796,3.478242,GO:0008010 [Name: structural constituent of ch...,proteoglycan 4-like
1,N5.HOG0002341,129227961,Loss,More likely in orbweavers,62.0,1.432432,0.235955,1.803488,GO:0004222 [Name: metalloendopeptidase activit...,astacin-like metalloprotease toxin 5
2,N5.HOG0004537,NaN,Loss,More likely in orbweavers,62.0,1.727273,0.182796,2.245930,NaN,NaN
3,N5.HOG0005532,NaN,Loss,More likely in orbweavers,85.0,0.384615,0.028037,2.618705,NaN,NaN
4,N5.HOG0006029,129222738,Loss,More likely in orbweavers,52.0,2.103448,0.428571,1.590876,-,arylsulfatase B-like
...,...,...,...,...,...,...,...,...,...,...
1578,N5.HOG0070810,NaN,Loss,More likely in non-orbweavers,53.0,0.232877,2.142857,-2.219386,NaN,NaN
1579,N5.HOG0070869,129230573,Loss,More likely in non-orbweavers,51.0,0.384615,1.820513,-1.554630,GO:0043130 [Name: ubiquitin binding]; GO:00431...,uncharacterized LOC129230573
1580,N5.HOG0071249,NaN,Loss,More likely in non-orbweavers,61.0,0.084337,1.682927,-2.993465,NaN,NaN
1581,N5.HOG0071283,129226255,Loss,More likely in non-orbweavers,51.0,0.267606,2.333333,-2.165539,-,uncharacterized LOC129226255


In [15]:
ort_res = pd.concat([loss_locs_filtered, dup_locs_filtered])
ort_res.to_excel(f'{results}/Supplementary_Table_4_OddsRatioTest_hits.xlsx', index=False)

## Spider accessions and BUSCO scores  

This code uses source spreadsheets that are not stored in this repo.

In [40]:
buscos = pd.read_csv(f'{assets}/orthorun_list_fams_busco90.csv')
accessions = pd.read_csv(f'{assets}/txptome_accessions.csv')
accessions.drop(columns=['Infraorder', 'Family', 'Web_Type', 'Cribellum'], inplace=True)

buscos_accessions = pd.merge(buscos, accessions, on='Species', how='left')

pd.set_option('display.max_columns', None, 'display.max_rows', None)

buscos_accessions.drop_duplicates(inplace=True)
buscos_accessions.reset_index(drop=True, inplace=True)
# Collapse rows: group by all columns except 'Accession', join 'Accession' values as comma-separated string
collapsed_buscos_accessions = buscos_accessions.groupby([col for col in buscos_accessions.columns if col != 'Accession'], dropna=False)['Accession'].apply(lambda x: ', '.join([str(i) for i in x if pd.notnull(i)])).reset_index()
collapsed_buscos_accessions.rename(columns={'Single': 'Single_copy_BUSCOs', 'Duplicated': 'Duplicated_BUSCOs', 'Total': 'Total_BUSCOs'}, inplace=True)
collapsed_buscos_accessions.to_excel(f'{results}/Supplementary_Table_1_SpiderAccessions_BUSCOs.xlsx', index=False)
collapsed_buscos_accessions


,Species,Infraorder,Family,Single_copy_BUSCOs,Duplicated_BUSCOs,Total_BUSCOs,Accession
0,Acroaspis_sp_IDV6688,Araneomorphae,Araneidae,2280,429,2709,IANO01
1,Acrosomoides_sp_IDV7426,Araneomorphae,Araneidae,2306,338,2644,ICOJ01
2,Ainerigone_saitoi,Araneomorphae,Linyphiidae,2238,550,2788,IBTE01
3,Ambicodamus_IDV6680,Araneomorphae,Nicodamidae,2261,409,2670,ICBW01
4,Antrodiaetus_roretzi,Mygalomorphae,Antrodiaetidae,2190,349,2539,IBPI01
5,Araneus_inustus,Araneomorphae,Araneidae,2261,385,2646,IAHS01
6,Arcuphantes_paiki,Araneomorphae,Linyphiidae,2217,483,2700,IAWN01
7,Arcuphantes_tamaensis,Araneomorphae,Linyphiidae,2326,433,2759,IAKU01
8,Arcuphantes_uenoi,Araneomorphae,Linyphiidae,2330,488,2818,IBBK01
9,Argiope_aetheroides,Araneomorphae,Araneidae,2344,375,2719,IASB01


In [39]:
# Count rows with NaN in the 'Accession' column before collapsing
num_nan_accession = buscos_accessions['Accession'].isna().sum()
print(f"Rows with NaN in 'Accession': {num_nan_accession}")

Rows with NaN in 'Accession': 10
